<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG302 - Geoquímica y Petrología** </span>

Bienvenido al curso!!!

Vamos a revisar ejemplos de <span style="color:gold">geoquímica</span> y <span style="color:gold">petrología</span> usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y geología general.

<span style="color:gold; font-size:20px">**Pyrolite** </span>

***
- [Geoquímica en Python](#parte-1)
- [Diagrama ternario usando `pyroplot.scatter`](#parte-2)
- [Clasificación QAP](#parte-3)
- [Clasificación TAS](#parte-4)

***

<a id="parte-1"></a>

### <span style="color:lightgreen">**Geoquímica en Python**</span>
***

La geoquímica utiliza la composición de las muestras para estudiar los materiales de la Tierra y sus procesos. Vamos a conservar los ejemplos originales de Pyrolite: dispersión y densidad ternarias, QAP y TAS. En cada caso revisaremos qué se midió y qué transformación necesita la figura.

| Entrada | Operación | Resultado |
| --- | --- | --- |
| Tres concentraciones en la misma unidad | Cerrar las tres partes a una suma de 1 | Una posición en el triángulo |
| Porcentajes modales de minerales | Cerrar Q, A y P a 100 | Un campo petrográfico de la plantilla QAP |
| Óxidos mayores de una roca volcánica | Recalcular la composición a 100 en base anhidra | Dos coordenadas para TAS |

En JupyterLite ejecuta la siguiente celda primero. El navegador usa Pyrolite 0.3.6 porque la versión de Matplotlib de este curso es 3.8.4. Para ejecutar localmente con Python 3.11 o posterior, puedes instalar las versiones probadas con `python -m pip install "pyrolite==0.3.9" "numpy==2.2.6" "matplotlib==3.11.2" "mpltern==1.0.5"`.

Si aparece un aviso sobre scikit-learn al importar Pyrolite, es una dependencia opcional. Aquí usamos SciPy para la densidad y los clasificadores por polígonos de Pyrolite.

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    import micropip

    await piplite.install(["numpy", "pandas", "matplotlib", "scipy", "sympy", "requests",
                           "mpltern==1.0.4", "periodictable==2.1.0", "tinydb==4.9.0"])
    # Pyrolite 0.3.6 importa el lector de docstrings de numpydoc.
    # Instalamos las dependencias de cálculo arriba; Sphinx no interviene en esta lección.
    await micropip.install(["numpydoc==1.8.0", "pyrolite==0.3.6"], deps=False)

In [ ]:
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import pyrolite.comp  # Registra el accesor de pandas .pyrocomp.
import pyrolite.geochem  # Registra .pyrochem.
from pyrolite.plot import pyroplot  # Registra .pyroplot.
from pyrolite.plot.templates import QAP, TAS
from pyrolite.util.classification import QAP as ClasificadorQAP
from pyrolite.util.classification import TAS as ClasificadorTAS

print(f"Pyrolite {version('pyrolite')} · Matplotlib {version('matplotlib')}")
data = pd.read_csv("files/rocas.csv")
data.sample(5, random_state=42)

<span style="color:gold">**Antes de graficar: comprueba unidades y especie química**</span>

Un valor de Mg elemental en % en masa puede convertirse a MgO equivalente multiplicando por la razón de masas molares. Con `renorm=False` Pyrolite conserva la escala de la entrada. Comprueba el resultado para 3% de Mg con masas molares redondeadas. Esta conversión expresa la misma cantidad de Mg como óxido equivalente.

In [ ]:
magnesio = pd.DataFrame({"Mg": [1.0, 3.0, 6.0]})
como_oxido = magnesio.pyrochem.convert_chemistry(to=["MgO"], renorm=False)
manual = magnesio["Mg"] * (24.305 + 15.999) / 24.305
pd.DataFrame({"Mg (%)": magnesio["Mg"], "MgO Pyrolite (%)": como_oxido["MgO"],
              "MgO a mano (%)": manual}).round(3)

<a id="parte-2"></a>

### <span style="color:lightgreen">**Diagrama ternario usando pyroplot.scatter**</span>
***

Usaremos SiO₂, MgO y hierro total expresado como FeO (`FeOT`) del archivo `rocas.csv`. El orden de las columnas se traduce en **arriba, izquierda y derecha**, en ese orden.

El triángulo representa la razón entre estas tres partes. Para SiO₂ dividimos `SiO2` por la suma de las tres columnas de esa fila. El valor cerrado de SiO₂ responde a «¿qué fracción de estas tres partes corresponde a sílice?». El porcentaje de SiO₂ de la roca completa es otra cantidad.

Comprobamos que las filas sean finitas y no negativas, y que su suma sea positiva. Conservamos las partes nulas para dispersión. Para densidad en coordenadas de log-razón necesitaremos partes estrictamente positivas. Tomamos una muestra reproducible de hasta 900 filas para que el ejemplo sea rápido.

In [ ]:
componentes = ["SiO2", "MgO", "FeOT"]
partes = data[componentes]
validas = (np.isfinite(partes).all(axis=1) & partes.ge(0).all(axis=1)
           & partes.sum(axis=1).gt(0))
print(f"Filas con tres partes válidas: {validas.sum():,} de {len(data):,}")

datos_validos = data.loc[validas]
muestra = datos_validos.sample(min(900, len(datos_validos)), random_state=42)
originales = muestra[componentes]
# Usamos renormalise para partes positivas y la división directa en el borde.
ternario = originales.div(originales.sum(axis=1), axis=0)
positivas = originales.gt(0).all(axis=1)
ternario.loc[positivas] = originales.loc[positivas].pyrocomp.renormalise(scale=1.0)
print(f"Filas para dispersión: {len(ternario)} | para densidad: {positivas.sum()}")

fila = originales.iloc[0]
cierre_manual = fila / fila.sum()
pd.DataFrame({"Original (% masa)": fila, "Cierre a mano": cierre_manual,
              "Cierre usado": ternario.iloc[0]}).round(4)

Cambia una parte en el primer registro y anticipa qué ocurrirá con las otras dos proporciones después de cerrar. Al multiplicar las tres partes por el mismo número, la posición se conserva.

En la figura, el primer panel muestra el triángulo completo. El segundo usa los mismos puntos y `set_ternary_lim` para acercarnos a la región rica en SiO₂. Los colores corresponden a la etiqueta en el archivo y permiten comparar la superposición de las muestras.

In [ ]:
colores = {"basalt": "#0072B2", "andesite": "#B84D12", "dacite": "#087E73", "rhyolite": "#8A4B9B"}
nombres = {"basalt": "Basalto", "andesite": "Andesita", "dacite": "Dacita", "rhyolite": "Riolita"}

def rotular_ternario(ax, arriba, izquierda, derecha):
    ax.set_tlabel(arriba)
    ax.set_llabel(izquierda)
    ax.set_rlabel(derecha)
    ax.grid(alpha=.22)
    ax.set_axisbelow(True)

fig, ejes = plt.subplots(1, 2, figsize=(10.1, 4.9), subplot_kw={"projection": "ternary"})
for ax in ejes:
    for roca, grupo in ternario.groupby(muestra["Nombre"]):
        grupo.pyroplot.scatter(ax=ax, s=12, marker="o", alpha=.58, color=colores[roca],
                               label=nombres[roca], axlabels=False)
    rotular_ternario(ax, "SiO₂", "MgO", "FeOT")
ejes[0].set_title("Composición de las tres partes", pad=14)
si_min = .65
ejes[1].set_ternary_lim(tmin=si_min, tmax=1., lmin=0., lmax=1.-si_min, rmin=0., rmax=1.-si_min)
ejes[1].set_title("Acercamiento · SiO₂ ≥ 0.65", pad=14)
handles, labels = ejes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, .08, 1, 1))
plt.show()

<span style="color:gold">**Densidad ternaria**</span>

`pyroplot.density` estima dónde se concentran las composiciones. Su estimación ternaria usa log-razones, por lo que filtramos partes nulas. Si una medición es cero por un límite de detección, documenta esa situación antes de considerar una imputación.

Mostraremos contornos que encierran aproximadamente el 90%, 66% y 33% de la **densidad estimada**. Pyrolite calcula esas regiones sobre una malla. Los niveles pueden variar con el suavizado, la selección de filas y el tamaño de la muestra. El acercamiento recorta la figura después de calcular la densidad.

In [ ]:
para_densidad = ternario.loc[positivas]
niveles = [0.90, 0.66, 0.33]
tintas = ["#0072B2", "#B84D12", "#087E73"]
estilos = ["-", "--", "-."]
fig, ax = plt.subplots(figsize=(6.6, 5.3), subplot_kw={"projection": "ternary"})
para_densidad.pyroplot.scatter(ax=ax, s=8, color="#526575", alpha=.22, axlabels=False)
para_densidad.pyroplot.density(ax=ax, mode="density", bins=36, contours=niveles,
                             percentiles=True, colors=tintas, linestyles=estilos,
                             linewidths=1.8, label_contours=False, axlabels=False)
ax.set_ternary_lim(tmin=si_min, tmax=1., lmin=0., lmax=1.-si_min, rmin=0., rmax=1.-si_min)
rotular_ternario(ax, "SiO₂", "MgO", "FeOT")
ax.set_title(f"Densidad estimada · {len(para_densidad)} composiciones", pad=15)
leyenda = [Line2D([0], [0], color=c, ls=s, lw=1.8, label=f"{p:.0%}")
           for p, c, s in zip(niveles, tintas, estilos)]
fig.legend(handles=leyenda, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, .08, 1, 1))
plt.show()

<a id="parte-3"></a>

### <span style="color:lightgreen">**Clasificación QAP**</span>
***

QAP usa **proporciones modales de minerales**: Q es cuarzo, A es feldespato alcalino y P es plagioclasa. Usaremos cuatro ejemplos plutónicos sintéticos que contienen cuarzo. Sus porcentajes modales suman 100 incluyendo M, agrupación de minerales máficos y otros minerales que quedan fuera de QAP. Para el triángulo recalculamos solo Q+A+P a 100.

Con datos reales, verifica primero si el diagrama corresponde al tipo de roca. Las rocas con feldespatoides y las altamente máficas requieren considerar otras partes del esquema IUGS. Las concentraciones de óxidos del archivo `rocas.csv` no dan por sí solas porcentajes modales medidos.

En JupyterLite, Pyrolite puede avisar que hay ceros al calcular los polígonos de QAP. Los vértices de la plantilla incluyen ceros para representar los bordes del triángulo. Las cuatro muestras se comprueban en la tabla.

In [ ]:
modal = pd.DataFrame(
    {"Q": [27., 26., 24., 22.], "A": [43., 33., 14., 3.],
     "P": [20., 29., 49., 56.], "M": [10., 12., 13., 19.]},
    index=["M1", "M2", "M3", "M4"],
)
assert np.isfinite(modal).all().all() and modal.ge(0).all().all()
assert np.allclose(modal.sum(axis=1), 100)
qap_original = modal[["Q", "A", "P"]]
assert qap_original.sum(axis=1).gt(0).all()
qap = qap_original.pyrocomp.renormalise(scale=100.)
# El modelo espera las columnas en orden Q, A, P y trabaja con fracciones.
qap_fracciones = qap / 100
qap_modelo = ClasificadorQAP()
campo_qap = qap_modelo.predict(qap_fracciones.to_numpy(), data_scale=1.)
campo_qap.index = qap.index
modal.join(qap.add_suffix("_cerrado")).assign(Campo=campo_qap).round(1)

Para M1, calcula cuánto vale Q al cerrar a 100. Después compara M1 con M4. Su porción de cuarzo cambia poco, pero la relación entre feldespato alcalino y plagioclasa cambia bastante. Observa dónde cae cada punto respecto de los límites de la plantilla.

In [ ]:
fig, ax = plt.subplots(figsize=(7.1, 5.4), subplot_kw={"projection": "ternary"})
QAP(ax=ax, linewidth=.8, color="#718293")
tonos = ["#0072B2", "#B84D12", "#087E73", "#8A4B9B"]
formas = ["o", "s", "D", "^"]
for (id_muestra, q), tono, forma in zip(qap_fracciones.iterrows(), tonos, formas):
    ax.scatter(*q, color=tono, marker=forma, s=70, edgecolors="white",
               zorder=4, label=f"{id_muestra} · {campo_qap[id_muestra]}")
rotular_ternario(ax, "Q · Cuarzo", "A · Feldespato alcalino", "P · Plagioclasa")
ax.set_title("QAP plutónico · fracciones modales", pad=16)
fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, .12, 1, 1))
plt.show()

<a id="parte-4"></a>

### <span style="color:lightgreen">**Clasificación TAS**</span>
***

TAS clasifica rocas volcánicas según SiO₂ y los álcalis totales (Na₂O + K₂O), ambos en % en masa sobre **base anhidra**. Usamos la variante `LeMaitre` de Pyrolite tanto para dibujar los campos como para obtener su código.

Antes de clasificar necesitamos saber qué óxidos forman el total anhidro y cómo se reportaron los volátiles. En tres ejemplos sintéticos completos calcularemos el porcentaje seco de cada óxido mayor. Primero quitamos H₂O y CO₂ de la suma. Después dividimos **cada** óxido por el total anhidro y multiplicamos por 100. Si en un ensayo faltan óxidos, cerrar solo lo que aparece en la tabla puede cambiar artificialmente la posición.

In [ ]:
mayores = ["SiO2", "TiO2", "Al2O3", "FeOT", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5"]
# Análisis sintéticos en % en masa, con los volátiles reportados aparte.
analisis = pd.DataFrame(
    [[48.0, 1.1, 15.1, 9.8, .15, 8.1, 10.1, 2.8, .7, .15, 3.3, .7],
     [55.68, .86, 16.23, 6.72, .1, 3.84, 7.19, 3.46, 1.73, .19, 3.4, .6],
     [70.56, .29, 13.23, 2.45, .05, .39, 1.37, 4.02, 5.49, .15, 1.7, .3]],
    columns=mayores + ["H2O", "CO2"], index=["V1", "V2", "V3"],
)
assert np.isfinite(analisis).all().all() and analisis.ge(0).all().all()
total_anhidro = analisis[mayores].sum(axis=1)
assert total_anhidro.gt(0).all()
seco = analisis[mayores].pyrocomp.renormalise(scale=100.)
assert np.allclose(seco.sum(axis=1), 100)

tas_seco = seco[["SiO2"]].assign(**{"Na2O + K2O": seco["Na2O"] + seco["K2O"]})
tas_original = analisis[["SiO2"]].assign(**{"Na2O + K2O": analisis["Na2O"] + analisis["K2O"]})
variante = "LeMaitre"
campo_tas = ClasificadorTAS(which_model=variante).predict(tas_seco, data_scale=100.)
tabla_tas = pd.DataFrame({
    "Total reportado": analisis.sum(axis=1), "Total anhidro original": total_anhidro,
    "SiO₂ original": tas_original["SiO2"], "SiO₂ seco": tas_seco["SiO2"],
    "Álcalis originales": tas_original["Na2O + K2O"], "Álcalis secos": tas_seco["Na2O + K2O"],
    "Campo TAS": campo_tas,
})
tabla_tas.round(2)

Al dividir por el total anhidro, ¿cuánto cambia la sílice de V2? Observa el desplazamiento de ambas coordenadas. El marcador hueco muestra la posición antes de recalcular y el sólido muestra la entrada al clasificador.

En esta variante, `Bs` es el campo del basalto subalcalino, `O2` el de la andesita y `R` el de la riolita.

Conservamos la nube original de andesitas para comparar posiciones. La base anhidra y los óxidos faltantes no están documentados en `rocas.csv`. Por eso sus posiciones se muestran tal como vienen y los códigos de clasificación se calculan solo para los análisis sintéticos completos.

In [ ]:
andesita = muestra.loc[muestra["Nombre"] == "andesite"].copy()
andesita["Na2O + K2O"] = andesita["Na2O"] + andesita["K2O"]

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True, sharey=True)
for ax in ejes:
    TAS(ax=ax, which_model=variante, add_labels=True, which_labels="ID",
        linewidth=.7, color="#7A8795", fontsize=8)
    ax.set(xlim=(40, 80), ylim=(0, 15), xlabel="SiO₂ (% en masa)",
           ylabel="Na₂O + K₂O (% en masa)")
    ax.grid(alpha=.16)
    ax.set_axisbelow(True)
izq, der = ejes
izq.scatter(andesita["SiO2"], andesita["Na2O + K2O"], s=12, color="#526575", alpha=.48)
izq.set_title("Andesitas del curso · datos reportados")
der.set_title("Tres análisis sintéticos · ajuste a base anhidra")
for id_muestra, color in zip(tas_seco.index, tonos):
    antes, despues = tas_original.loc[id_muestra], tas_seco.loc[id_muestra]
    der.annotate("", xy=tuple(despues), xytext=tuple(antes),
                 arrowprops={"arrowstyle": "->", "color": color, "lw": 1.3})
    der.scatter(*antes, s=48, facecolors="white", edgecolors=color, zorder=3)
    der.scatter(*despues, s=55, color=color, edgecolors="white", zorder=4)
    der.annotate(f"{id_muestra} · {campo_tas[id_muestra]}", xy=tuple(despues), xytext=(5, 9),
                 textcoords="offset points", color=color, fontsize=9, weight="bold")
fig.tight_layout()
plt.show()

**Comprueba tu lectura.** Explica por qué la suma usada para cerrar el ternario SiO₂–MgO–FeOT es distinta de la suma anhidra usada en TAS. Cambia los volátiles de un ejemplo manteniendo fijas las razones entre óxidos anhidros. Anticipa qué coordenadas se desplazan y cuáles quedan iguales.

Pyrolite conecta tablas, transformaciones y figuras. Para aplicar los ejemplos a una campaña, revisa primero unidades, datos faltantes y el método que generó el análisis. Después comprueba qué columnas necesita el método y cómo se interpretan las filas cercanas a un límite.

[Pyrolite, conversión de elementos y óxidos](https://pyrolite.readthedocs.io/en/main/examples/geochem/convert_chemistry.html) · [Densidad y contornos](https://pyrolite.readthedocs.io/en/develop/examples/plotting/density.html) · [Clasificación TAS](https://pyrolite.readthedocs.io/en/main/examples/util/TAS.html)